In [2]:
import pandas as pd
import folium
from folium.plugins import HeatMap

CHICAGO_LATLON = [41.8781, -87.6298]

rides = pd.read_csv('../data/rides.csv', parse_dates=['ride_start', 'ride_end'], dtype={'pickup_zip': 'str', 'dropoff_zip': 'str'})
rides = rides.query('ride_type not in ("Connect Express", "Connect Saver", "Delivery", "Package Express")')
rides = rides.sort_values('ride_start')

# Show the range of ZIP codes I picked up or dropped off in on a map of Chicago
rides_by_pickup_zip = rides \
                        .groupby(['pickup_zip'])['ride_start'] \
                        .count() \
                        .reset_index(name='ride_count') \
                        .rename(columns={'pickup_zip': 'zip'})
rides_by_dropoff_zip = rides \
                        .groupby(['dropoff_zip'])['ride_start'] \
                        .count() \
                        .reset_index(name='ride_count') \
                        .rename(columns={'dropoff_zip': 'zip'})
combined = pd.concat([rides_by_pickup_zip, rides_by_dropoff_zip])
total_rides_by_zip = combined.groupby('zip', as_index=False)['ride_count'].sum()
zipcodes = pd.read_csv('../data/zip_code_to_lat_long.csv', dtype={'zip': 'str'})
total_rides_by_zip = total_rides_by_zip \
                        .merge(zipcodes, on='zip', how='left')[['zip', 'ride_count', 'latitude', 'longitude']] \
                        .dropna(subset=['latitude', 'longitude'])
print(total_rides_by_zip)

chicago_map = folium.Map(location=CHICAGO_LATLON, zoom_start=10)
heat_data = [[row['latitude'], row['longitude'], row['ride_count']] for index, row in total_rides_by_zip.iterrows()]
HeatMap(heat_data, radius=15, max_opacity=0.8).add_to(chicago_map)

for index, row in total_rides_by_zip.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Zip: {row['zip']}<br>Count: {row['ride_count']}",
        tooltip=row['zip'],
        icon=folium.DivIcon(html=f"""<div style="font-family: courier new; color: blue">{row['zip']}</div>""")
    ).add_to(chicago_map)

chicago_map.save('../visualizations/chicago_map.html')

       zip  ride_count   latitude  longitude
1    46394           1  41.678891 -87.500609
2    60004           1  42.111995 -87.979921
3    60005           1  42.065588 -87.983624
4    60007           1  42.005441 -88.013215
5    60008           1  42.072481 -88.022440
..     ...         ...        ...        ...
140  60714          14  42.033787 -87.819500
141  60803           6  41.673159 -87.726135
142  60804          12  41.845914 -87.762193
143  60805           5  41.720134 -87.702920
144  60827           1  41.650162 -87.630667

[142 rows x 4 columns]
